### RecursiveCharacterTextSplitter

- Document를 설정한 chunk를 기준으로 분리


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


### Vector DB

- TextSplitter로 분리한 데이터를 Embedding 처리한 후 저장


In [9]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

API_KEY = os.getenv("NVIDIA")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

embedding = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FIASS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


documents :  191
chunk :  332
FIASS 생성완료
결과 :  0
페이지 :  31

것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 자금은 예금자보호대상이 아닙니다. 
그러나 펀드의 경우 투자자들의 자금으로 취득한 펀드재산은 자산운용회사의 고유재산과 분리되어 
신탁업자가 별도로 관리하기 때문에 자산운용회사가 파산하더라도 펀드내의 집합투자재산은 안전하다고 
할 수 있습니다.

결과 :  1
페이지 :  28

ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기

### RecursiveCharacterTextSplitter 및 Vector DB로 LLM까지 연결


In [14]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(api_key=API_KEY, model=MODEL)

embeddings = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        ),
    ]
)

parser = StrOutputParser()

# question = "펀드란 무엇인가요?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


result :  제공된 문서에서 답을 찾을 수 없습니다.


### Vector DB에서 similarity_search_with_score로 통한 유사도 점수 확인


In [17]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print("질문 : ", question)

    for i, (doc, score) in enumerate(result):
        print("검색 결과")
        print("distance : ", score)
        print("page : ", doc.metadata.get("page"))
        print("내용 : ", doc.page_content[:500])

python-dotenv could not parse statement starting at line 30


/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


질문 :  펀드란 무엇인가요?
검색 결과
distance :  1.2219472
page :  31
내용 :  것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 
검색 결과
distance :  1.2478447
page :  28
내용 :  ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 

### MMR(Maximum Marginal Relevance)

- 질문과 관련 있으면서 서로 다른 정보를 가진 문서를 검색


In [18]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드의 종류와 특징은 무엇인가요?"

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

similarity_docs = similarity_retriever.invoke(question)

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5,
    },
)

mmr_docs = mmr_retriever.invoke(question)

print("\n")
print("=" * 80)
print("Similarity Search 결과")
print("=" * 80)

for i, doc in enumerate(similarity_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])


# ============================================================
# 11. MMR 결과
# ============================================================

print("\n")
print("=" * 80)
print("MMR Search 결과")
print("=" * 80)

for i, doc in enumerate(mmr_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(




Similarity Search 결과

----- 결과 1 -----
Page: 28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 투자 목표 등을 검토해 본 후 가입하는 것이 좋습니다. 
•투자성향이란? :  수익 및 투자위험에 대한 본인의 기대 수준을 말합니다. 높은 수익을 위해서 손실이 발생해도
감내할 수 있는지, 아니

----- 결과 2 -----
Page: 30
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집30
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
다. 적합한 펀드 선정
•투자권유 청취 및 상품선정 : 투자자 유형에 대한 분류결과에 기초하여 본인에게 적정한 투자권유를 받고 적합 
한 상품을 선정합니다. 인터넷을 통해 가입할 경우 전자서명의 방법으로 확인서에 서명하는 방식으로 가입이 진행 
됩니다. 투자권유를 원치 않거나 본인 투자유형 등급보다 높은 등급의 펀드투자를 원할 경우 투자자 확인서에 
서명 후 투자 가능합니다.
라. 펀드에 대한 설명 청취
•펀드에 대한 설명 청취 : 투자권유 펀드의 투자대상자산 등 운용전략, 원본손실위험 등 투자위험, 보수·수수료
및 펀드운용비용, 환매방법 등에 대한 설명을 판매회사의 직원에게 듣습니다. 
마. 투자자 의사확인 후 가입절차
•판매회

### Metadata Filtering

- Vector DB에서 검색하기 전에 특정 조건에 맞는 문서만 검색 대상으로 제한


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("NVIDIA")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)


documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드란 무엇인가?"

all_results = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(all_results):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print("Content:")
    print(doc.page_content[:500])

filtered_results = vectorstore.similarity_search(question, k=3, filter={"page": 50})

print()

if not filtered_results:
    print("조건에 맞는 문서를 찾지 못했습니다.")

else:
    for i, doc in enumerate(filtered_results):
        print(f"\n----- 결과 {i + 1} -----")

        print("Page:", doc.metadata.get("page"))

        print("Content:")
        print(doc.page_content[:500])

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(



----- 결과 1 -----
Page: 31
Content:
것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 

----- 결과 2 -----
Page: 28
Content:
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 투자 목표 등을 검토해 본 후 가입하는 것이 좋습니다. 
•투자

### Hybrid Search

- Vector Search(의미가 비슷한 문서 반환)과 Keyword Search(정확한 단어로 검색)를 모두 활용


In [8]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"


llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)


vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 3

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


result :  모바일폰을 사용해 금융거래를 할 때는 다음과 같은 사항을 꼭 유의해야 합니다.

1. **공식 배포처 확인**  
   금융회사가 안내하는 공식 배포처를 통해서만 금융서비스를 이용합니다.  
2. **금융정보 저장 금지**  
   모바일폰이나 인터넷에 금융정보를 저장하지 않도록 합니다.  
3. **비밀번호 관리**  
   금융거래 비밀번호를 안전하게 관리하고, 정기적으로 변경합니다.  
4. **분실·도난 시 즉시 중지**  
   모바일폰이 분실하거나 도난당한 경우, 모바일폰 금융서비스 사용을 즉시 중지합니다.  
5. **교체·수리 전 정보 삭제**  
   모바일폰을 교체하거나 수리하기 전에는 중요 정보를 반드시 삭제합니다.  
6. **SMS·OTP 활용**  
   휴대폰 문자서비스(SMS)나 일회용비밀번호(OTP)를 이용해 보안을 강화합니다.  
7. **사용환경 변경 금지**  
   모바일폰 사용환경을 임의로 변경하지 않도록 주의합니다.  
8. **보안업데이트 & 바이러스 검사**  
   모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사를 실시합니다.  
9. **잠금기능 설정 & 비밀번호 주기적 변경**  
   모바일폰에 ‘잠금기능’을 설정하고 ‘잠금비밀번호’를 수시로 변경합니다.  
10. **비보안 무선랜 주의**  
    출처가 불분명하거나 보안 설정이 없는 Wi‑Fi를 사용할 때는 주의합니다.  

위 10가지 항목을 지키면 모바일폰을 통한 금융거래 시 보안을 크게 강화할 수 있습니다.


### Reranker

- 찾아온 문서들 중에서 질문과 관련 있는 순서대로 다시 정리
- 일반적으로 Retriever보다 계산량이 많기에 후보를 추려오면 그 뒤 사용


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA, NVIDIARerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
NVIDIA_RERANK_KEY = os.getenv("NVIDIA_RERANK_KEY")
NVIDIA_RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

reranker = NVIDIARerank(model=NVIDIA_RERANK_MODEL, api_key=NVIDIA_RERANK_KEY)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)

print("검색된 후보 문서 : ", len(retrieved_docs))

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

final_docs = reranked_docs[:3]

for i, doc in enumerate(final_docs, start=1):
    print(f"\n--- 최종 문서 {i} ---")

    print("page:", doc.metadata.get("page"))

    print(doc.page_content[:500])


context = "\n\n".join([doc.page_content for doc in final_docs])

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]\n{question}",
        ),
    ]
)

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_25966/2685781352.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


검색된 후보 문서 :  18

--- 최종 문서 1 ---
page: 25
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
 금융투자상품에 투자하기   25
① 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기 
② 모바일폰이나 인터넷에 금융정보를 저장하지 않기
③ 금융거래 비밀번호를 안전하게 관리하기
④ 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기.
⑤ 모바일폰 교체·수리 전 중요 정보 삭제하기
⑥ 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기
⑦ 모바일폰 사용환경을 임의로 변경하지 않기
⑧ 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기
⑨ 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기
⑩ 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용시 주의하기
모바일폰 금융거래 10계명
모바일폰은 언제, 어디에서나 사용하는 필수품이 되었습니다. 이제는 금융거래도 모바일폰을 이용하여 언제, 
어디서든지 할 수 있습니다.  
여러 금융기관에서는 금리

--- 최종 문서 2 ---
page: 16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
나날이 복잡해져 가는 금융투자상품에 대해 정확
하게 이해하기 위하여 투자설명서를 꼼꼼히 체크
하여야 하며, 이해가 가지 않는 부분이 있다면 금
융투자회사 직원에게 설명을 요청하는 것이 좋습
니다.
특히, 계좌개설 시 반드시 투자설명서 및 약관을 
교부받아 적어도 투자기간 동안 보관해 두어야 합
니다.
금융투자상품 가입 시 작성하는 서류에 투자자 본
인이 자필로 서명하거나 기재하는 문구가 여러가
지 있습니다. 특히 투자설명서의 이해 여부를 확인하기 위한 본인기재사항은 투자자가 해당 금융투자상품을 
이해하고 투자한다는 것을 선언하는 것으로 사후 법적다툼이 발생할 때 판단하는 1차적인 자료로 활용될 수 
있으므로 그 내용을 충분히 확인한 후 작성하는 

### Multi-Query

- 하나의 질문을 여러 개의 검색 질문으로 변환한 뒤 각각 검색하는 방법
- 벡터 검색은 의미가 비슷하면 어느 정도 찾아주지만, 한 번의 검색으로는 놓치는 문서가 생길 수 있습니다.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
NVIDIA_RERANK_KEY = os.getenv("NVIDIA_RERANK_KEY")
NVIDIA_RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwarg={"k": 3})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

prompt = ChatPromptTemplate.from_template("""
    다음 질문에 대해 문서 검색에 사용할 수 있는 서로 다른 검색 질문 3래를 만들어주세요.

    원래 질문
    {question}

    조건:
    - 원래 질문의 의미는 유지할 것
    - 서로 다른 표현을 사용할 것
    - 검색에 도움이 되는 핵심 단어를 포함할 것
    - 번호나 설명없이 검색 질문만 한 줄씩 출력할 것
""")

parser = StrOutputParser()

reranker = NVIDIARerank(model=NVIDIA_RERANK_MODEL, api_key=NVIDIA_RERANK_KEY)

chain = prompt | llm | parser

generated_queries_text = chain.invoke({"question": question})

generated_queries = [
    query.strip() for query in generated_queries_text.split("\n") if query.strip()
]

all_docs = []

for i, query in enumerate(generated_queries, start=1):
    print(query)
    docs = hybrid_retriever.invoke(query)

    print("검색된 문서 수:", len(docs))

    # 검색 결과를 하나의 리스트에 모읍니다.
    all_docs.extend(docs)


print("\n전체 검색 결과:", len(all_docs))

unique_docs = {}

for doc in all_docs:
    page = doc.metadata.get("page")

    if page not in unique_docs:
        unique_docs[page] = doc
unique_docs = list(unique_docs.values())

print("중복 제거 후 문서 수 : ", len(unique_docs))

reranked_docs = reranker.compress_documents(documents=unique_docs, query=question)

final_docs = reranked_docs[:3]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


모바일 폰으로 금융 거래할 때 주의해야 할 사항은 무엇인가요?
검색된 문서 수: 14
모바일 뱅킹 시 안전 수칙 및 위험 요소는 무엇인가?
검색된 문서 수: 14
핸드폰을 이용한 금융 거래에서 보안 관점에서 주의할 점은?
검색된 문서 수: 14

전체 검색 결과: 42
중복 제거 후 문서 수 :  31
result :  모바일 폰을 이용해 금융 거래를 할 때는 다음과 같은 점에 유의해야 합니다.

| 순번 | 유의사항 | 상세 내용 |
|------|-----------|-----------|
| ① | 공식 배포처 확인 | 금융회사가 안내하는 공식 앱·웹사이트를 이용하세요. 비공식 채널은 피합니다. |
| ② | 금융정보 비저장 | 모바일·인터넷에 금융정보(계좌번호, 비밀번호 등)를 저장하지 말고 필요 시마다 직접 입력하거나 안전하게 복구하세요. |
| ③ | 비밀번호 안전 관리 | 반복되는 숫자(0000·1111)나 전화번호·생년월일 등 쉽게 추측되는 비밀번호는 피하고, 정기적으로 바꾸세요. |
| ④ | 분실·도난 시 즉시 중지 | 폰이 분실·도난되면 즉시 금융 서비스 이용을 중지하고, 해당 기관에 신고하세요. |
| ⑤ | 교체·수리 전 삭제 | 모바일 장치를 교체하거나 수리하는 경우 중요 정보를 완전히 삭제하세요. |
| ⑥ | OTP 또는 SMS 활용 | 일회용 비밀번호(OTP)나 SMS를 통한 인증을 적극 활용해 보안을 강화하세요. |
| ⑦ | 사용환경 그대로 유지 | 기기로서의 사용환경(설정, 앱)을 임의로 바꾸지 않도록 주의하세요. |
| ⑧ | 보안 업데이트·바이러스 검사 | 보안 업데이트를 정기적으로 적용하고 안티바이러스로 검사하세요. |
| ⑨ | 잠금 기능 및 비밀번호 관리 | 툴 비활성화 기능(잠금)을 설정하고, 잠금 비밀번호는 수시로 변경하세요. |
| ⑩ | 무선랜 주의 | 출처가 불분명하거나 보안설정이 없는 Wi‑Fi(공공 와이파이)에 연결 시 주의하고, 가능하면 금융 거래 시 사용을 자제하세요. |

- **공공장소 Wi‑Fi

### Query Transformation

- 사용자 질문을 그대로 검색하지 않고 검색하기 좋은 형태로 바꿔주는 것
- 1. Rewrite, Step-back, 3. HyDE 등


In [5]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"


llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwarg={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

question = "그거 모바일로 금융거래 할 때 조심해야 하는 게 뭐였지?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

final_docs = retrieved_docs[:3]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


모바일폰으로 금융거래를 할 때는 다음과 같은 사항을 꼭 지켜야 합니다.  

| 주의 내용 | 구체적 행동 |
|---|---|
| **공신력 있는 채널을 이용** | 금융회사가 안내하는 공식 배포처(지점, 홈페이지, HTS 등)를 통해 서비스 사용 |
| **정보 저장 금지** | 모바일폰이나 인터넷에 금융 정보(계좌비밀번호, 거래비밀번호 등)를 저장해서는 안 됨 |
| **비밀번호 안전관리** | 금융거래 비밀번호를 안전하게 보관하고, 3세제(전문가 증명서 등)와 별도로 사용 |
| **분실·도난 시 즉시 중지** | 모바일폰을 분실·도난당했을 때는 금융서비스 이용을 즉시 중지하고 ①-④ 항목에 따라 비밀번호를 포함한 모든 정보를 해제 |
| **교체·수리 전 삭제** | 모바일폰을 교체하거나 수리하기 전에는 중요 정보를 반드시 삭제 |
| **보안 인증 수단 활용** | SMS 또는 OTP 같은 일회용 비밀번호를 이용하고, 공인인증서를 하드디스크에 저장할 때는 PC지정서비스 이용 |
| **환경 변형 금지** | 모바일폰을 무단으로 설정을 변경하거나, 보안 설정을 해지해서는 안 됨 |
| **정기적 보안 업데이트** | 모바일폰 보안업데이트를 정기적으로 수행하고, 바이러스 검사를 실행 |
| **잠금 기능 설정** | 모바일폰 ‘잠금기능’을 설정하고, ‘잠금비밀번호’는 주기적으로 변경 |
| **공공 와이‑파이 주의** | 출처가 불분명하거나 보안 설정이 없는 무선랜(Wi‑Fi) 사용 시 주의, 비밀번호가 없거나 공개된 와이‑파이는 피하고 비밀번호가 있는 보안된 와이‑파이만 사용 |

이 외에도 **공공장소에서 비밀번호가 없거나, 보안이 취약한 Wi‑Fi를 사용할 때는 해킹 위험이 높아지므로 특별히 주의**해야 합니다.  

> **제공된 문서에서 답을 찾을 수 없습니다.**


### Context Compression

- 검색된 문서에서 질문에 필요한 부분만 뽑아내는 것


In [4]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
NVIDIA_RERANK_KEY = os.getenv("NVIDIA_RERANK_KEY")
NVIDIA_RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로 다시 작성하세요.

    조건:
    - 원래 질문의 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

reranked_docs = reranked_docs[:5]

compressor = LLMChainExtractor.from_llm(llm)

compressed_docs = []

for doc in reranked_docs:
    compressed_doc = compressor.compress_documents(documents=[doc], query=question)

    compressed_docs.extend(compressed_doc)

final_docs = compressed_docs[:5]

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


모바일 폰을 사용한 금융 거래 시 주의해야 할 사항은 다음과 같습니다.

| 번호 | 주의 사항 | 핵심 포인트 |
|------|----------|------------|
| ① | 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기 | 공식 앱이 외부에서 만든 가짜 앱인지 확인 |
| ② | 모바일폰이나 인터넷에 금융정보를 저장하지 않기 | 비밀번호, 계좌번호 등을 로컬 저장 금지 |
| ③ | 금융거래 비밀번호를 안전하게 관리하기 | 비밀번호는 복잡하게 설정하고 주기적으로 변경 |
| ④ | 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기 | 즉시 서비스 비활성화 |
| ⑤ | 모바일폰 교체·수리 전 중요 정보 삭제하기 | 재설치 전에 모든 금융 관련 데이터를 완전히 삭제 |
| ⑥ | 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기 | 2단계 인증·OTP 사용 |
| ⑦ | 모바일폰 사용환경을 임의로 변경하지 않기 | 설정을 임의로 바꾸면 보안 위험 증가 |
| ⑧ | 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기 | 최신 보안 패치와 안티바이러스 사용 |
| ⑨ | 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기 | 기기 잠금 활성화 후 비밀번호 자주 바꾸기 |
| ⑩ | 출처가 불분명하거나 보안설정 없는 무선랜(Wi‑Fi) 사용 시 주의하기 | 안정적이고 암호화된 Wi‑Fi 사용 권장 |

또한, 새로운 내용으로 **금융사기에 대한 예방·대처법**과 **모바일 폰을 활용한 투자 시 유의사항**이 추가되었으므로, 금융사기 위험을 최소화하고 투자 시 신중히 확인하도록 안내합니다.


### Citation

- 답변의 근거가 어디인지 같이 보여주는 것


In [5]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings, NVIDIARerank
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
RERANK_API_KEY = os.getenv("NVIDIA_RERANK_KEY")
RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

reranker = NVIDIARerank(api_key=RERANK_API_KEY, model=RERANK_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로
    다시 작성하세요.

    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.

    사용자 질문:
    {question}
    """
)

parser = StrOutputParser()

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewrite_chain = rewrite_prompt | llm | parser

rewritten_question = rewrite_chain.invoke({"question": question})

print("Rewrite:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

reranked_docs = reranked_docs[:5]

context_parts = []

for doc in reranked_docs:
    page = doc.metadata.get("page", "unknown")
    source = doc.metadata.get("source", "unknown")
    context_parts.append(
        f"""
        [출처: p.{page}]
        [파일: {source}]
        {doc.page_content}
        """
    )


context = "\n\n".join(context_parts)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로
            질문에 답변하는 도우미입니다.

            반드시 제공된 Context만 근거로 답변하세요.

            답변에 사용한 문서의 출처를
            반드시 다음 형식으로 표시하세요.

            [출처: p.페이지번호]

            Context에 없는 내용은 추측하지 마세요.

            문서에서 답을 찾을 수 없다면:

            "제공된 문서에서 답을 찾을 수 없습니다."

            라고 답변하세요.

            [문서 Context]
            {context}
            """,
        ),
        (
            "user",
            """
            [질문]
            {question}
            """,
        ),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"question": question, "context": context})

print()
print("result : ", result)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


Rewrite:
모바일 금융거래를 할 때 주의해야 할 보안 및 안전 수칙은?

result :  모바일 금융거래 시 주의해야 할 사항은 다음과 같습니다.

1. **공식 배포처 확인**  
   - 금융회사가 안내하는 공식 앱·웹사이트를 이용합니다.  
2. **모바일에 금융정보 저장 금지**  
   - 스마트폰에 거래·계좌 정보를 저장하지 않습니다.  
3. **비밀번호 안전 관리**  
   - 반복 숫자(0000, 1111)나 개인정보(생일, 휴대폰 번호 등)를 사용하지 말고 주기적으로 바꿉니다.  
4. **분실·도난 시 즉시 중지**  
   - 분실·도난 시 해당 기기의 금융서비스 사용을 바로 중지합니다.  
5. **교체·수리 전 데이터 삭제**  
   - 모바일 교체·수리 전 중요한 정보를 완전히 삭제합니다.  
6. **문자·OTP 활용**  
   - SMS·OTP(일회용 비밀번호) 서비스를 이용해 두 차원 인증합니다.  
7. **사용환경 임의 변경 금지**  
   - 사전 승인 없이 설정을 변경하지 않습니다.  
8. **보안 업데이트와 바이러스 검사**  
   - 정기적으로 보안 패치를 받고 바이러스 스캔을 수행합니다.  
9. **잠금 기능 및 비밀번호 관리**  
   - ‘잠금 기능’을 설정하고, 잠금비밀번호는 자주 변경합니다.  
10. **불분명한 Wi‑Fi 주의**  
    - 출처가 불분명하거나 보안 설정이 없는 무선랜(Wi‑Fi)은 사용을 피합니다.  

> **출처: p.25**


### RAG Evaluation(평가)

- 필요한 문서를 제대로 가져왔는지 평가


In [4]:
import os

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
    NVIDIARerank,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

NVIDIA_RERANK_KEY = os.getenv("NVIDIA_RERANK_KEY")
NVIDIA_RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")

llm = ChatNVIDIA(
    model=MODEL,
    api_key=API_KEY,
)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)


PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)


vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})


bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10


hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def evaluate_hit_rate(
    question,
    relevant_pages,
    k=5,
):
    rewritten_question = rewrite_chain.invoke({"question": question})

    retrieved_docs = hybrid_retriever.invoke(rewritten_question)

    reranked_docs = reranker.compress_documents(
        documents=retrieved_docs,
        query=question,
    )

    top_docs = reranked_docs[:k]

    retrieved_pages = [doc.metadata.get("page") for doc in top_docs]

    hit = any(page in relevant_pages for page in retrieved_pages)

    return hit, retrieved_pages, rewritten_question


evaluation_data = [
    {
        "question": "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?",
        "relevant_pages": [25, 16, 21],
    },
]
hits = []

for item in evaluation_data:
    hit, pages, rewritten_question = evaluate_hit_rate(
        question=item["question"],
        relevant_pages=item["relevant_pages"],
        k=5,
    )

    hits.append(int(hit))

    print("=" * 60)

    print("원래 질문:")
    print(item["question"])

    print("\nRewrite된 질문:")
    print(rewritten_question)

    print("\n정답 페이지:")
    print(item["relevant_pages"])

    print("\n검색된 Top 5 페이지:")
    print(pages)

    print("\nHit@5:")
    print(int(hit))

hit_rate = sum(hits) / len(hits)

print("\n" + "=" * 60)

print(f"최종 Hit Rate@5: {hit_rate:.4f}")

print("=" * 60)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


원래 질문:
모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?

Rewrite된 질문:
모바일 금융거래 시 주의사항은?

정답 페이지:
[25, 16, 21]

검색된 Top 5 페이지:
[25, 4, 16, 24, 142]

Hit@5:
1

최종 Hit Rate@5: 1.0000


### LLM-as-a-Judge

- LLM에게 다른 LLM의 답변을 채점 시키는 것

### Context Relevance

- 검색해서 가져온 Context가 질문에 관련되어 있는지 체크

### Faithfulness

- 환각(Hallucination)을 얼마나 하고 있는지 확인

### Answer Relevance

- 질문에 제대로 답변 했는지 확인


In [5]:
import os

from dotenv import load_dotenv

from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
    NVIDIARerank,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

NVIDIA_RERANK_KEY = os.getenv("NVIDIA_RERANK_KEY")
NVIDIA_RERANK_MODEL = os.getenv("NVIDIA_RERANK_MODEL")


llm = ChatNVIDIA(
    model=MODEL,
    api_key=API_KEY,
)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

reranker = NVIDIARerank(
    model=NVIDIA_RERANK_MODEL,
    api_key=NVIDIA_RERANK_KEY,
)

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

반드시 아래 제공된 문서 내용만 근거로 답변하세요.

문서에 답이 없다면
"제공된 문서에서 답을 찾을 수 없습니다."
라고 답변하세요.

[문서 내용]
{context}
""",
        ),
        ("user", "[질문]\n{question}"),
    ]
)

answer_chain = answer_prompt | llm | StrOutputParser()

context_eval_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 검색 품질 평가자입니다.

사용자 질문과 검색된 문서를 보고,
검색된 문서가 질문에 얼마나 관련 있는지 평가하세요.

점수:
0.0 = 전혀 관련 없음
0.5 = 일부 관련
1.0 = 매우 관련 있음

반드시 숫자 하나만 출력하세요.

[질문]
{question}

[Context]
{context}
"""
)

context_eval_chain = context_eval_prompt | llm | StrOutputParser()

faithfulness_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 답변의 사실성을 평가하는 평가자입니다.

답변에 포함된 내용이 제공된 Context에 의해
뒷받침되는지를 평가하세요.

Context에 없는 내용을 답변이 추가했다면
점수를 낮게 주세요.

점수:
0.0 = 대부분 근거 없음
0.5 = 일부만 근거 있음
1.0 = 모든 내용이 Context에 근거함

반드시 숫자 하나만 출력하세요.

[Context]
{context}

[Answer]
{answer}
"""
)

faithfulness_chain = faithfulness_prompt | llm | StrOutputParser()

answer_relevance_prompt = ChatPromptTemplate.from_template(
    """
당신은 RAG 답변의 질문 관련성을 평가하는 평가자입니다.

사용자의 질문과 답변을 비교하여
답변이 질문에 얼마나 적절하게 답했는지 평가하세요.

점수:
0.0 = 질문에 답하지 못함
0.5 = 일부만 답함
1.0 = 질문에 정확하게 답함

반드시 숫자 하나만 출력하세요.

[질문]
{question}

[Answer]
{answer}
"""
)

answer_relevance_chain = answer_relevance_prompt | llm | StrOutputParser()

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewritten_question = rewrite_chain.invoke({"question": question})

print("=" * 60)

print("원래 질문:")
print(question)

print("\nRewrite 질문:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(
    documents=retrieved_docs,
    query=question,
)

reranked_docs = reranked_docs[:5]

context = "\n\n".join(doc.page_content for doc in reranked_docs)

answer = answer_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)


print("\n" + "=" * 60)

print("최종 Answer:")
print(answer)

context_score = context_eval_chain.invoke(
    {
        "question": question,
        "context": context,
    }
)

faithfulness_score = faithfulness_chain.invoke(
    {
        "context": context,
        "answer": answer,
    }
)

answer_relevance_score = answer_relevance_chain.invoke(
    {
        "question": question,
        "answer": answer,
    }
)

print("\n" + "=" * 60)

print("RAG Evaluation")

print("-" * 60)

print("Context Relevance:", context_score)

print("Faithfulness:", faithfulness_score)

print("Answer Relevance:", answer_relevance_score)

print("=" * 60)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


원래 질문:
모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?

Rewrite 질문:
모바일 금융 거래 시 주의사항은?

최종 Answer:
모바일 금융거래 시 주의해야 할 주요 사항은 다음과 같습니다.

| # | 주의사항 | 핵심 내용 |
|---|----------|-----------|
| ① | **공식 배포처 확인** | 금융회사가 안내하는 공식 앱·웹에서만 거래를 진행한다. |
| ② | **모바일에 금융정보 저장 금지** | 거래에 필요한 데이터(계좌정보·비밀번호)는 기기에 저장하지 않는다. |
| ③ | **비밀번호 안전 관리** | 4자리 연속(0000, 1111 등)·핸드폰번호·생일 등 추측이 쉬운 비밀번호 사용 금지; 주기적으로 변경. |
| ④ | **분실·도난 시 즉시 모바일 금산서비스 중지** | 분실·도난 즉시 ‘모바일폰 금융서비스 사용 중지’를 수행한다. |
| ⑤ | **교체·수리 전 중요정보 삭제** | 모바일폰 교체·수리 전 보관한 금융정보를 완전히 삭제한다. |
| ⑥ | **SMS·OTP 이용** | 문자/SMS, 일회용 비밀번호(OTP)를 활용해 보안 강화. |
| ⑦ | **수동으로 환경 변경 금지** | 모바일폰 사용 환경을 임의로 변경하지 않는다. |
| ⑧ | **보안 업데이트와 바이러스 검사** | 보안 업데이트를 정기적으로 수행하고 바이러스 검사를 실행한다. |
| ⑨ | **잠금기능·잠금비밀번호 설정** | ‘잠금기능’을 설정하고, 잠금비밀번호는 수시로 변경한다. |
| ⑩ | **공용 Wi‑Fi 주의** | 출처가 불분명하거나 보안 설정이 없는 무선랜(Wi‑Fi)을 사용 시 주의한다. |

> **추가 팁**  
> - 모니터링 서비스 (통장/비밀번호 신용, SMS 알림 등)를 활용해 거래 내역을 즉시 확인한다.  
> - 은행·증권사 콜센터 번호를 메모해 두고, 전산장애 시에는 **“홈페이지·콜센터에 즉시 연락하여 주문 의사(종목·가격·시기)를 명시**하고, PC 화면 캡처 등 증거를 보관한다.  